# MATH-500 Clean Accuracy Run

**What this notebook does, in plain terms:** runs a proper sample of MATH-500 problems through the pilot model at natural (uncapped) thinking length, and checks how often the model gets the right answer. This is NOT a drift run. Its only job is to prove two things before we build anything on top:

1. The answer extractor (`math500_matcher.py`) is trustworthy, we're actually reading the model's real answer, not mis-grading it.
2. Accuracy lands near the known reference (Mirage reports ~83% on MATH-500 for this exact model). If it does, the pipeline is sound and we can move on to the short-vs-long drift work.

**Why a big token budget:** MATH-500 problems are harder than GSM8K. In an earlier run at 4096 tokens, ~45% of problems never finished thinking and looked "wrong" when they were really just cut off. We use a large budget here so truncation stops contaminating the accuracy number.

**Files that must be uploaded into this same folder first:** `harness.py` (v0.3.4), `math500_matcher.py`, `math500_worker.py`. Run cells top to bottom.

## 1. Install packages
One-time per fresh kernel.

In [ ]:
!pip install -q transformers accelerate datasets


## 2. Version check — do this FIRST, every time

This has bitten us repeatedly: an edited file on disk doesn't update an already-loaded module, and a stale copy silently produces wrong results. This cell confirms the harness on disk is the fully-fixed v0.3.4 AND that the matcher's key fixes are present, before any GPU time is spent. If either line is wrong, fix the upload before continuing.

In [ ]:
import harness, math500_matcher, inspect
print("harness version:", harness.HARNESS_VERSION, "(want 0.3.4)")
print("harness has LaTeX-noise strip:", hasattr(harness, "_strip_latex_noise"), "(want True)")
src = inspect.getsource(math500_matcher)
print("matcher has brace-less frac fix:", "_FRAC_BARE" in src, "(want True)")
print("matcher has nested-frac fix:", "_FRAC_BRACED" in src, "(want True)")


## 3. Pick the sample

Sets which MATH-500 problems to run. Start with 60 for a solid accuracy read (bigger than the earlier n=40, small enough to finish quickly across 7 GPUs). Fixed seed so the sample is reproducible.

In [ ]:
import random
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
N = 60
BUDGET = 16384   # large, so truncation stops eating results on hard items
idxs = list(range(len(ds)))
random.Random(0).shuffle(idxs)
all_idxs = idxs[:N]
print(f"selected {len(all_idxs)} problems, budget {BUDGET} tokens")


## 4. Smoke test — 2 GPUs, 1 item each

Cheap insurance before the full launch: confirms the workers actually run and both exit cleanly. GPU 0 is skipped on purpose (the notebook kernel's own model, if any, holds it; and it keeps the pattern consistent with the full run).

In [ ]:
import subprocess, os, time

test_procs = []
for gpu_id, idx in zip([1, 2], all_idxs[:2]):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    log = open(f"acc_test_gpu{gpu_id}.log", "w")
    p = subprocess.Popen(
        ["python", "math500_worker.py", "--gpu", str(gpu_id),
         "--indices", str(idx), "--budget", str(BUDGET), "--out", f"acc_test_gpu{gpu_id}.json"],
        env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    test_procs.append((gpu_id, p, log))
    print(f"launched test on gpu {gpu_id} for idx {idx}")

while any(p.poll() is None for _, p, _ in test_procs):
    time.sleep(10)
for gpu_id, p, log in test_procs:
    print(f"gpu {gpu_id}: exit code {p.returncode}  (0 = good)")


## 5. Full launch — GPUs 1 through 7

Splits the sample round-robin across 7 worker GPUs, one independent process each. Only run this once the smoke test above showed exit code 0 on both.

In [ ]:
WORKER_GPUS = list(range(1, 8))
shards = [all_idxs[i::len(WORKER_GPUS)] for i in range(len(WORKER_GPUS))]

procs = []
for gpu_id, shard in zip(WORKER_GPUS, shards):
    if not shard:
        continue
    idx_str = ",".join(str(i) for i in shard)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    log = open(f"acc_shard_gpu{gpu_id}.log", "w")
    p = subprocess.Popen(
        ["python", "math500_worker.py", "--gpu", str(gpu_id),
         "--indices", idx_str, "--budget", str(BUDGET), "--out", f"acc_shard_gpu{gpu_id}.json"],
        env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    procs.append((gpu_id, p, log))
    print(f"launched gpu {gpu_id} with {len(shard)} items")
print(f"\n{len(procs)} workers running across GPUs {WORKER_GPUS}")


## 6. Check progress
Safe to re-run any time; just reports status without disturbing the workers.

In [ ]:
done = sum(1 for _, p, _ in procs if p.poll() is not None)
print(f"{done}/{len(procs)} workers finished")
for gpu_id, p, _ in procs:
    print(f"  gpu {gpu_id}: {'done' if p.poll() is not None else 'running'}")


## 7. Merge and report

Run once all workers show "done". Combines every shard, reports overall accuracy, the match-tier breakdown, and, importantly, how many items truncated (`pred=None`). If truncation is high even at this budget, that's real signal about the model, not a grading problem.

In [ ]:
import json
from collections import Counter

all_results = []
for gpu_id, p, log in procs:
    try:
        all_results.extend(json.load(open(f"acc_shard_gpu{gpu_id}.json")))
    except FileNotFoundError:
        print(f"WARNING: no output for gpu {gpu_id}, check acc_shard_gpu{gpu_id}.log")

with open("math500_accuracy_run.json", "w") as f:
    json.dump(all_results, f, indent=2)

n = len(all_results)
n_correct = sum(r["correct"] for r in all_results)
n_trunc = sum(1 for r in all_results if r["pred"] is None)
n_finished = n - n_trunc

print(f"Total items:            {n}")
print(f"Truncated (pred=None):  {n_trunc}")
print(f"Accuracy (all):         {n_correct}/{n} = {n_correct/n:.3f}")
if n_finished:
    fin_correct = sum(r['correct'] for r in all_results if r['pred'] is not None)
    print(f"Accuracy (finished):    {fin_correct}/{n_finished} = {fin_correct/n_finished:.3f}")
print(f"Reference (Mirage, 8B, MATH-500): ~0.832")
print("Match-tier breakdown:", Counter(r['tier'] for r in all_results))


## 8. Hand-check the mismatches

The matcher is normalization-based, not a symbolic solver, so any `no_match` that isn't a truncation needs a human glance to confirm it's a real model error and not a matcher gap. This lists them; read each pred vs gold and decide.

In [ ]:
to_check = [r for r in all_results if r["tier"] == "no_match" and r["pred"] is not None]
print(f"{len(to_check)} non-truncation mismatches to hand-check:\n")
for r in to_check:
    print(f"idx={r['math500_idx']} lvl={r.get('level')} subj={r.get('subject')}")
    print(f"   pred: {r['pred']!r}")
    print(f"   gold: {r['gold']!r}\n")
